In [1]:
import polars as pl
import polars.selectors as cs
import pandas as pd
import numpy as np
import glob
import os
import io
from hta.trace_analysis import TraceAnalysis

In [9]:
analyzer = TraceAnalysis({0: "device.0.json"}, "/proj/threadtune-PG0/amir/kineto_traces/bert-inference")
rank = 0
trace_data = analyzer.t.get_trace(rank)
sym = analyzer.t.symbol_table.get_sym_table()

Parsed /proj/threadtune-PG0/amir/kineto_traces/bert-inference/device.0.json time = 0.02 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425


Parsed /proj/threadtune-PG0/amir/kineto_traces/bert-inference/device.0.json backend=ParserBackend.JSON in 0.07 seconds; current PID:231144
Overall parsing of /proj/threadtune-PG0/amir/kineto_traces/bert-inference/device.0.json in 0.09 seconds; current PID:231144
leaving parse_multiple_ranks duration=0.09 seconds
leaving parse_traces duration=0.09 seconds
There is only one iteration in the trace. The analysis result may not be accurate.


In [15]:
df = (
    pl.from_pandas(trace_data[["index", "ts", "dur", "cat", "name"]])
    .with_columns(
        pl.col(col).map_elements(lambda x: sym[x]) for col in ["cat", "name"]
    )
)

In [29]:
df["cat"].unique()

cat
str
"""gpu_user_annotation"""
"""cuda_sync"""
"""kernel"""
"""gpu_memcpy"""
"""cuda_runtime"""
"""user_annotation"""
"""overhead"""
"""cpu_op"""


In [31]:
kernel_df = df.filter(pl.col("cat") == "kernel")

In [37]:
(
    kernel_df.join(kernel_df, how="cross")
    .filter(
        (pl.col("index") != pl.col("index_right")) &
        pl.col("ts").is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right")) &
        (pl.col("ts") + pl.col("dur")).is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right"))
    )
    .is_empty()
)

True

In [34]:
kernel_df["dur"].sum()

1601.0

In [18]:
cpu_df = df.filter(pl.col("cat") == "cpu_op")

In [21]:
cpu_roots_df = cpu_df.join(
    cpu_df.join(cpu_df, how="cross")
    .filter(
        (pl.col("ts") > pl.col("ts_right")) &
        (pl.col("ts") + pl.col("dur") < pl.col("ts_right") + pl.col("dur_right"))
    ),
    on="index",
    how="anti"
)

In [47]:
cpu_leaves_df = cpu_df.join(
    cpu_df.join(cpu_df, how="cross")
    .filter(
        (pl.col("ts") < pl.col("ts_right")) &
        (pl.col("ts") + pl.col("dur") > pl.col("ts_right") + pl.col("dur_right"))
    ),
    on="index",
    how="anti"
)
cpu_leaves_df

index,ts,dur,cat,name
i16,i64,f64,str,str
3,101069,61.0,"""cpu_op""","""aten::as_strided"""
5,101230,14.0,"""cpu_op""","""aten::as_strided"""
7,101317,14.0,"""cpu_op""","""aten::as_strided"""
10,101484,20.0,"""cpu_op""","""aten::view"""
12,101600,38.0,"""cpu_op""","""aten::empty"""
…,…,…,…,…
1317,311702,1522.0,"""cpu_op""","""aten::tanh"""
1321,313354,30.0,"""cpu_op""","""aten::empty_strided"""
1323,313427,21.0,"""cpu_op""","""aten::empty_strided"""


In [38]:
(
    cpu_roots_df.join(cpu_roots_df, how="cross")
    .filter(
        (pl.col("index") != pl.col("index_right")) &
        pl.col("ts").is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right")) &
        (pl.col("ts") + pl.col("dur")).is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right"))
    )
    .is_empty()
)

True

In [22]:
cpu_roots_df["dur"].sum()

204052.0

In [42]:
kernel_df.join(
    kernel_df.join(cpu_roots_df, how="cross")
    .filter(
        pl.col("ts").is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right")) &
        (pl.col("ts") + pl.col("dur")).is_between(pl.col("ts_right"), pl.col("ts_right") + pl.col("dur_right"))
    ),
    on="index",
    how="anti"
).is_empty()

True

In [46]:
df.filter(pl.col("cat") == "cuda_runtime")["dur"].sum()

158759.0

In [45]:
cpu_roots_df["dur"].sum()

204052.0